# MCP Server Blueprint — Build Your Own Domain Agent

This notebook is a **step-by-step guide** to creating a complete set of MCP (Model Context Protocol) servers that power a domain-specific AI agent through the Spotfire Agent harness. By the end, you'll have 4 production-ready MCP servers that any MCP-compatible client can orchestrate.

---

## What You'll Build

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              USER QUESTION                                   │
│              "Why is my equipment underperforming?"                          │
└──────────────────────────────────┬──────────────────────────────────────────┘
                                   │
                                   ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                     SPOTFIRE AGENT HARNESS                                   │
│                                                                             │
│   ┌──────────────┐     ┌──────────────┐                                    │
│   │  SKILLS.md   │     │  Agents.md   │                                    │
│   │ (Workflows & │     │ (Identity &  │                                    │
│   │  Routing)    │     │  Config)     │                                    │
│   └──────────────┘     └──────────────┘                                    │
└──────────────────────────────────┬──────────────────────────────────────────┘
                                   │ MCP Protocol (Streamable HTTP)
                                   │
          ┌────────────────────────┼────────────────────────┐
          │                        │                        │
          ▼                        ▼                        ▼
┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
│  UC Functions    │  │   Genie Space    │  │    AI Search     │
│  MCP Server      │  │   MCP Server     │  │    MCP Server    │
│                  │  │                  │  │                  │
│  AI diagnostic   │  │  NL queries over │  │  Semantic search │
│  functions that  │  │  your tables -   │  │  over reference  │
│  interpret data  │  │  no SQL needed   │  │  documents       │
│  (5-10 tools)    │  │  by the user     │  │                  │
└──────────────────┘  └──────────────────┘  └──────────────────┘
          │                                            │
          └──────────────────┐  ┌──────────────────────┘
                             │  │
                             ▼  ▼
                  ┌──────────────────┐
                  │  Databricks SQL  │
                  │  MCP Server      │
                  │                  │
                  │  Schema discovery│
                  │  + SQL fallback  │
                  └──────────────────┘
```

---

## The 4 MCP Servers

| # | Server | What It Does | MCP Endpoint Pattern |
|---|--------|-------------|---------------------|
| 1 | **UC Functions** | AI-powered domain expertise (diagnosis, assessment, forecasting) | `/api/2.0/mcp/functions/{catalog}/{schema}` |
| 2 | **Genie Space** | Natural language queries over your data tables | `/api/2.0/mcp/genie/{space_id}` |
| 3 | **AI Search** | Semantic retrieval over reference documents | `/api/2.0/mcp/ai-search/{catalog}/{schema}` |
| 4 | **Databricks SQL** | Schema discovery and complex SQL fallback | `/api/2.0/mcp/sql` |

---

## How It Works

1. **SKILLS.md** defines *what* the agent can do — tool routing rules, composition workflows, and domain knowledge
2. **Agents.md** defines *who* the agent is — identity, connected skills, MCP server configuration
3. The **Spotfire Agent harness** reads both files at deployment time and becomes a domain expert
4. User questions are routed to the right MCP server(s) based on the SKILLS.md decision tree
5. Multiple servers can be composed in a single turn (e.g., Genie retrieves data → UC Function interprets it)

---

## Prerequisites

- Databricks workspace with Unity Catalog enabled
- At least one domain dataset (1-3 tables with meaningful data)
- A schema where you can create functions and tables
- A Vector Search endpoint (or ability to create one)
- Basic understanding of your domain (what questions do experts ask?)

---

## What You Need to Provide

| Input | Description | Example |
|-------|------------|--------|
| Domain tables | Your data in Unity Catalog | `my_catalog.my_schema.sensor_data` |
| Domain knowledge | What an expert knows about your field | "Vibration > 7 mm/s is concerning" |
| Reference documents | Standards, procedures, best practices (PDF or markdown) | ISO standards, SOPs |
| Expert tasks | 5-10 things a domain expert does | Diagnose issues, assess health, forecast degradation |

---

## Steps Overview

| Step | Action | Cells |
|------|--------|-------|
| 1 | Configure variables | Cell 3 |
| 2 | Verify prerequisites | Cell 4 |
| 3-4 | Create & verify UC Functions | Cells 5-7 |
| 5 | Create Genie Space | Cells 8-9 |
| 6-7 | Generate & index reference docs | Cells 10-13 |
| 8 | Grant permissions | Cells 14-16 |
| 9 | Behavior Pack authoring guide | Cell 17 |
| 9b | Generate complete behavior pack | Cell 18 |
| 10 | Deploy the behavior pack | Cell 19 |
| 11 | Test MCP connectivity | Cell 20 |
| 12 | Summary & next steps | Cell 21 |

---

## 💡 Pro Tip: Use the Databricks Assistant

**Steps 3 through 7** (creating UC Functions, Genie Space, generating reference documents, and indexing them for AI Search) can be accomplished interactively with the help of the **Databricks Assistant (Genie Code)**.

Simply describe what you need in natural language:
- *"Create a UC function that diagnoses production line defects based on sensor readings and defect codes"*
- *"Create a Genie space with my tables for natural language querying"*
- *"Generate reference documents about pharmaceutical manufacturing best practices"*
- *"Index my reference documents in the volume into a Vector Search index"*

The assistant will generate the SQL/Python code, create the cells, execute them, and troubleshoot any issues — all within this notebook. This is especially useful if you're unfamiliar with `ai_query()` syntax or Vector Search configuration.

> **To use:** Open the Assistant chat panel (right sidebar) and describe your domain, tables, and the expert tasks you want to automate. The assistant will handle the implementation details.

# ✅ Quick-Start Checklist

Use this checklist to track your progress. Each item maps to a notebook cell you can run.

---

## Before You Begin
- [ ] Have at least 1-3 domain tables loaded in Unity Catalog
- [ ] Know your catalog and schema names
- [ ] Have a Vector Search endpoint available (or permissions to create one)
- [ ] Have a service principal created (or admin access to create one)
- [ ] Know your domain — what questions do your experts answer daily?

## Setup (5 minutes)
- [ ] **Cell 3** — Edit configuration variables (catalog, schema, tables, domain name)
- [ ] **Cell 4** — Run prerequisites check (all green ✅)

## Build MCP Servers (30-60 minutes)
- [ ] **Cells 5-6** — Create 5-10 UC Functions using templates *(or ask the Assistant!)*
- [ ] **Cell 7** — Verify functions are MCP-ready (all have COMMENTs)
- [ ] **Cells 8-9** — Create Genie Space, paste space_id back into Cell 3 *(or ask the Assistant!)*
- [ ] **Cells 10-11** — Generate reference documents *(or ask the Assistant!)*
- [ ] **Cells 12-13** — Index documents into Vector Search *(or ask the Assistant!)*

## Permissions (5 minutes)
- [ ] **Cell 15** — Run SQL grants for service principal
- [ ] **Cell 16** — Grant CAN_RUN on Genie Space

## Behavior Pack (15-30 minutes)
- [ ] **Cell 17** — Review pack structure, best practices, and authoring guidance
- [ ] **Cell 18** — Generate the complete behavior pack (pack.yaml + system_prompt.md + AGENTS.md + help.md + 4 skills)
- [ ] Customize `AGENTS.md` with your domain knowledge (thresholds, formulas, terms)
- [ ] Verify `uc-functions/SKILL.md` description lists ALL your function names + trigger verbs
- [ ] Verify `genie-data-questions/SKILL.md` includes table schemas (column names and types)
- [ ] Update `allowed-tools` in each skill with your actual tool names (space-separated!)

## Validate (5 minutes)
- [ ] **Cell 20** — Test MCP connectivity (all 4 servers green ✅)
- [ ] **Cell 21** — Review summary and note your MCP endpoint URLs

## Deploy Spotfire Agent
- [ ] Export the pack from the UC Volume (see Appendix or README for methods: CLI, UI, API)
- [ ] Hand the exported pack folder to the deployment team (mount as PVC/ConfigMap/image layer)
- [ ] Provide the 7 environment variables (4 MCP URLs + pack dir + OAuth credentials)
- [ ] Verify: agent card shows your name/description
- [ ] Verify: typing "help" returns your help.md text verbatim
- [ ] Test with a sample domain question end-to-end

---

> **⏱️ Estimated total time:** 1-2 hours for a complete setup (less with Assistant help).
> Most time is spent on UC Function design — the rest is largely automated.

In [0]:
# =============================================================================
# CONFIGURATION — Edit these values for your domain
# =============================================================================
# All subsequent cells reference these variables. Set them once here.
# =============================================================================

# --- Unity Catalog Location ---
# Where your functions, tables, and indexes will live
CATALOG = "my_catalog"                    # Your UC catalog name
SCHEMA = "my_schema"                      # Schema for functions + chunks table

# --- Domain Tables ---
# The tables containing your domain data (used by Genie Space)
# Add/remove as needed — list all tables the agent should query
DOMAIN_TABLES = [
    f"{CATALOG}.{SCHEMA}.my_primary_table",     # Your main fact/event table
    f"{CATALOG}.{SCHEMA}.my_dimension_table",   # Supporting dimension/lookup table
    # f"{CATALOG}.{SCHEMA}.my_timeseries_table", # Uncomment if you have time series
]

# --- Domain Description ---
# Used to generate reference docs and in the SKILL.md
DOMAIN_NAME = "My Domain"                 # e.g., "Energy Operations", "Semiconductor Manufacturing"
DOMAIN_DESCRIPTION = """Describe your domain in 2-3 sentences. What does the data represent?
What decisions do domain experts make? What questions do they ask?
"""

# --- Genie Space ---
GENIE_SPACE_NAME = f"{DOMAIN_NAME} Agent"  # Name for the Genie space
GENIE_SPACE_ID = ""                        # Filled after creation (Step 5)

# --- AI Search / Vector Index ---
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/reference_docs"  # Where reference docs live (for indexing)
PACK_OUTPUT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/behavior_packs"  # Where generated packs are saved (separate from docs)
AI_SEARCH_ENDPOINT = "my-ai-search-endpoint"  # Your Vector Search endpoint name
EMBEDDING_MODEL = "databricks-gte-large-en"   # Embedding model (this is the default)
INDEX_NAME = f"{CATALOG}.{SCHEMA}.{DOMAIN_NAME.lower().replace(' ', '_')}_reference_index"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.{DOMAIN_NAME.lower().replace(' ', '_')}_reference_chunks"
MAX_CHUNK_CHARS = 1500                         # Target chunk size for indexing

# --- Service Principal ---
# The service principal that external clients (Spotfire Agent) use to access MCP
SERVICE_PRINCIPAL_ID = "00000000-0000-0000-0000-000000000000"  # Replace with your SP UUID

# --- AI Model ---
# Model used inside UC Functions for domain reasoning
AI_MODEL = "databricks-claude-sonnet-4"   # Or: databricks-meta-llama-3-3-70b-instruct

# --- Derived Values (don't edit) ---
FUNCTIONS_MCP_URL = f"/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"
GENIE_MCP_URL = f"/api/2.0/mcp/genie/{GENIE_SPACE_ID}" if GENIE_SPACE_ID else "/api/2.0/mcp/genie/<space_id>"
AI_SEARCH_MCP_URL = f"/api/2.0/mcp/ai-search/{CATALOG}/{SCHEMA}"  # Schema-scoped: auto-discovers all indexes
SQL_MCP_URL = "/api/2.0/mcp/sql"

print(f"""
{'='*70}
CONFIGURATION SUMMARY
{'='*70}
  Domain:           {DOMAIN_NAME}
  Catalog.Schema:   {CATALOG}.{SCHEMA}
  Tables:           {len(DOMAIN_TABLES)} table(s)
  Reference Docs:   {VOLUME_PATH}
  Behavior Packs:   {PACK_OUTPUT_PATH}
  AI Search:        {AI_SEARCH_ENDPOINT}
  Index:            {INDEX_NAME}
  Service Principal:{SERVICE_PRINCIPAL_ID}
  AI Model:         {AI_MODEL}
{'='*70}
MCP Endpoints (after setup):
  UC Functions:     {FUNCTIONS_MCP_URL}
  Genie Space:      {GENIE_MCP_URL}
  AI Search:        {AI_SEARCH_MCP_URL}
  Databricks SQL:   {SQL_MCP_URL}
{'='*70}
""")

In [0]:
# =============================================================================
# VERIFY PREREQUISITES
# =============================================================================
# Checks that your catalog, schema, tables, and volume are accessible.
# Creates the volume if it doesn't exist.
# =============================================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
errors = []

print("Checking prerequisites...\n")

# 1. Catalog exists and is accessible
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    print(f"  ✅ Catalog '{CATALOG}' is accessible")
except Exception as e:
    errors.append(f"Catalog '{CATALOG}' not accessible: {e}")
    print(f"  ❌ Catalog '{CATALOG}' not accessible")

# 2. Schema exists
try:
    spark.sql(f"USE SCHEMA {SCHEMA}")
    print(f"  ✅ Schema '{CATALOG}.{SCHEMA}' exists")
except Exception as e:
    errors.append(f"Schema '{SCHEMA}' not found")
    print(f"  ❌ Schema '{CATALOG}.{SCHEMA}' not found")
    print(f"     Create it with: CREATE SCHEMA {CATALOG}.{SCHEMA}")

# 3. Tables are accessible
for table in DOMAIN_TABLES:
    try:
        count = spark.sql(f"SELECT COUNT(*) AS n FROM {table}").collect()[0]["n"]
        print(f"  ✅ Table '{table}' accessible ({count:,} rows)")
    except Exception as e:
        errors.append(f"Table '{table}' not accessible")
        print(f"  ❌ Table '{table}' not accessible: {str(e)[:80]}")

# 4. Volumes exist (create if not)
for vol_path, vol_purpose in [(VOLUME_PATH, "reference docs"), (PACK_OUTPUT_PATH, "behavior packs")]:
    try:
        dbutils.fs.ls(vol_path)
        print(f"  ✅ Volume '{vol_path}' exists ({vol_purpose})")
    except Exception:
        print(f"  ⚠️  Volume not found — creating ({vol_purpose})...")
        try:
            vol_name = vol_path.split('/')[-1]
            spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{vol_name}")
            print(f"  ✅ Volume '{vol_path}' created")
        except Exception as e:
            errors.append(f"Cannot create volume '{vol_path}': {e}")
            print(f"  ❌ Cannot create volume: {e}")

# 5. Check Vector Search endpoint
try:
    vs_endpoints = w.vector_search_endpoints.list_endpoints()
    found = any(ep.name == AI_SEARCH_ENDPOINT for ep in vs_endpoints)
    if found:
        print(f"  ✅ Vector Search endpoint '{AI_SEARCH_ENDPOINT}' exists")
    else:
        print(f"  ⚠️  Vector Search endpoint '{AI_SEARCH_ENDPOINT}' not found")
        print(f"     Available endpoints: {[ep.name for ep in vs_endpoints]}")
        errors.append(f"Vector Search endpoint '{AI_SEARCH_ENDPOINT}' not found")
except Exception as e:
    print(f"  ⚠️  Could not list Vector Search endpoints: {str(e)[:80]}")

# Summary
print()
if errors:
    print(f"⚠️  {len(errors)} issue(s) to resolve before continuing:")
    for err in errors:
        print(f"    • {err}")
else:
    print("✅ All prerequisites met — ready to build MCP servers!")

# Step 3: Create UC Functions

## What Are UC Functions for MCP?

Unity Catalog SQL functions become **tools** that MCP clients can discover and call. When you create a function in a schema, it automatically appears as a callable tool at:

```
/api/2.0/mcp/functions/{catalog}/{schema}
```

The MCP client sees the function's `COMMENT` as the tool description, and each parameter's `COMMENT` as the parameter description. This is how the agent knows *when* and *how* to use each function.

## The Pattern: `ai_query()` + Structured Prompts

Each function uses `ai_query()` to call a foundation model with a domain-expert system prompt:

```sql
CREATE OR REPLACE FUNCTION catalog.schema.my_function(
    required_param STRING COMMENT 'Description for MCP clients',
    optional_param STRING DEFAULT '' COMMENT 'Optional context'
)
RETURNS STRING
COMMENT 'What this tool does - shown to MCP clients as the tool description'
RETURN ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
        'You are a domain expert. ',
        'Analyze: ', required_param, ' ',
        CASE WHEN optional_param != '' THEN CONCAT('Context: ', optional_param) ELSE '' END,
        ' Output raw JSON starting with { and ending with }.'
    ),
    modelParameters => named_struct('max_tokens', 500, 'temperature', 0.2)
);
```

## Design Principles

1. **Rich COMMENT metadata** — The function COMMENT and parameter COMMENTs are how the agent decides which tool to use. Make them descriptive.
2. **Required parameters first** — SQL requires all DEFAULT parameters come AFTER non-DEFAULT ones.
3. **JSON output** — Always instruct the model to return structured JSON for parseability.
4. **Low temperature** — Use 0.1-0.3 for consistent, factual responses.
5. **Appropriate max_tokens** — 400-600 for most assessments; up to 1000 for detailed reports.
6. **Domain persona** — Start the prompt with "You are a [domain] expert" for best results.

## How to Design Functions for Your Domain

Think about what a domain expert does:

| Expert Task | Function Pattern | Example |
|------------|-----------------|--------|
| Diagnose a problem | `diagnose_X(entity, symptoms, context)` | `diagnose_equipment_event(equip_id, event_type, sensors)` |
| Assess health/quality | `assess_X_health(entity, metrics)` | `assess_compressor_health(equip_id, vibration, pressure)` |
| Compare entities | `compare_X(entity_a, metrics_a, entity_b, metrics_b)` | `compare_plant_performance(plant_a, kpis_a, plant_b, kpis_b)` |
| Predict/forecast | `forecast_X(entity, trend, threshold)` | `forecast_degradation(equip_id, trend_values, alarm_limit)` |
| Classify/categorize | `classify_X(entity, observations)` | `classify_defect_mechanism(fail_bins, zone_name)` |
| Recommend actions | `recommend_X(entity, situation)` | `recommend_maintenance_action(equip_id, events, health)` |
| Interpret patterns | `interpret_X(entity, pattern_data)` | `interpret_sensor_anomaly(sensor, anomaly_type, value)` |
| Generate SPC/stats | `generate_spc_X(metric, values, limits)` | `generate_spc_assessment(metric, values, ucl, lcl)` |

**Aim for 5-10 functions** that cover the core expert tasks in your domain.

---

> **💡 Assistant Shortcut:** Instead of writing these functions manually, you can ask the Databricks Assistant to create them for you. Just describe the expert task in plain language:
>
> *"Create a UC function in my_catalog.my_schema that diagnoses pump failures based on vibration readings, discharge pressure, and flow rate. It should return JSON with root_cause, severity, and recommended_actions."*
>
> The assistant will generate the full `CREATE OR REPLACE FUNCTION` statement with proper `ai_query()` prompting, `COMMENT` metadata, parameter ordering, and JSON output instructions — then execute it directly. This also works for Steps 4–7 (Genie Space, reference docs, and AI Search indexing).

In [0]:
%sql
-- =============================================================================
-- UC FUNCTION TEMPLATES
-- =============================================================================
-- Copy and customize these templates for your domain.
-- Replace {{catalog}}, {{schema}}, {{domain}}, and customize the prompts.
-- =============================================================================

-- ─────────────────────────────────────────────────────────────────────────────
-- TEMPLATE 1: DIAGNOSTIC / ROOT-CAUSE ANALYSIS
-- Use when: User asks "why did X happen?", "what caused Y?"
-- ─────────────────────────────────────────────────────────────────────────────

/*
CREATE OR REPLACE FUNCTION {{catalog}}.{{schema}}.diagnose_issue(
    entity_id STRING COMMENT 'Identifier for the entity being diagnosed (e.g., equipment ID, patient ID, order ID)',
    entity_type STRING COMMENT 'Type/category of the entity (e.g., "compressor", "cardiac", "logistics")',
    issue_description STRING COMMENT 'Description of the issue or event to diagnose',
    severity STRING COMMENT 'Severity level: Low, Medium, High, Critical',
    duration DOUBLE COMMENT 'Duration of the issue in hours',
    supporting_data STRING DEFAULT '' COMMENT 'Additional context: measurements, readings, or observations (format: "key:value,key:value")',
    recent_history STRING DEFAULT '' COMMENT 'Summary of recent related events or trends'
)
RETURNS STRING
COMMENT 'Performs expert root-cause analysis on an issue or event. Analyzes the entity type, issue description, severity, and supporting data to identify probable causes, assess risk, and recommend immediate actions. Returns JSON with: root_cause, confidence, risk_level, mechanism, contributing_factors, immediate_actions, follow_up_investigations.'
RETURN ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
        'You are a senior {{domain}} expert performing root-cause analysis. ',
        'Diagnose this issue and provide structured analysis.\n\n',
        'Entity: ', entity_id, ' (Type: ', entity_type, ')\n',
        'Issue: ', issue_description, '\n',
        'Severity: ', severity, '\n',
        'Duration: ', CAST(duration AS STRING), ' hours\n',
        CASE WHEN supporting_data != '' THEN CONCAT('Data: ', supporting_data, '\n') ELSE '' END,
        CASE WHEN recent_history != '' THEN CONCAT('History: ', recent_history, '\n') ELSE '' END,
        '\nReturn ONLY valid JSON with fields: root_cause (string), confidence (HIGH/MEDIUM/LOW), ',
        'risk_level (LOW/MEDIUM/HIGH/CRITICAL), mechanism (string), contributing_factors (array), ',
        'immediate_actions (array), follow_up_investigations (array).\n',
        'Output raw JSON starting with { and ending with }.'
    ),
    modelParameters => named_struct('max_tokens', 500, 'temperature', 0.2)
);
*/

-- ─────────────────────────────────────────────────────────────────────────────
-- TEMPLATE 2: COMPARISON / BENCHMARKING
-- Use when: User asks "how does A compare to B?", "which is better?"
-- ─────────────────────────────────────────────────────────────────────────────

/*
CREATE OR REPLACE FUNCTION {{catalog}}.{{schema}}.compare_entities(
    entity_a_id STRING COMMENT 'Identifier for the first entity to compare',
    entity_a_metrics STRING COMMENT 'Key metrics for entity A (format: "metric1:value1,metric2:value2")',
    entity_b_id STRING COMMENT 'Identifier for the second entity to compare',
    entity_b_metrics STRING COMMENT 'Key metrics for entity B (format: "metric1:value1,metric2:value2")',
    entity_type STRING DEFAULT '' COMMENT 'Type of entities being compared (e.g., "plant", "region", "product line")',
    comparison_focus STRING DEFAULT '' COMMENT 'Specific aspect to focus comparison on (e.g., "reliability", "efficiency", "cost")'
)
RETURNS STRING
COMMENT 'Compares two entities across multiple performance metrics. Identifies which entity performs better overall, highlights specific gaps, and recommends best practices to transfer. Returns JSON with: winner_overall, performance_gaps, strengths_a, strengths_b, recommendations, best_practices_to_transfer.'
RETURN ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
        'You are a {{domain}} benchmarking expert. Compare these two entities and provide actionable insights.\n\n',
        'Entity A: ', entity_a_id, '\n  Metrics: ', entity_a_metrics, '\n',
        'Entity B: ', entity_b_id, '\n  Metrics: ', entity_b_metrics, '\n',
        CASE WHEN entity_type != '' THEN CONCAT('Type: ', entity_type, '\n') ELSE '' END,
        CASE WHEN comparison_focus != '' THEN CONCAT('Focus: ', comparison_focus, '\n') ELSE '' END,
        '\nReturn ONLY valid JSON with fields: winner_overall (string), score_a (number 0-100), ',
        'score_b (number 0-100), performance_gaps (array of {metric, gap, favors}), ',
        'recommendations (array), best_practices_to_transfer (array).\n',
        'Output raw JSON starting with { and ending with }.'
    ),
    modelParameters => named_struct('max_tokens', 500, 'temperature', 0.2)
);
*/

-- ─────────────────────────────────────────────────────────────────────────────
-- TEMPLATE 3: FORECASTING / PREDICTIVE
-- Use when: User asks "when will X fail?", "how long until Y?"
-- ─────────────────────────────────────────────────────────────────────────────

/*
CREATE OR REPLACE FUNCTION {{catalog}}.{{schema}}.forecast_degradation(
    entity_id STRING COMMENT 'Identifier for the entity being assessed',
    entity_type STRING COMMENT 'Type of entity (determines degradation model)',
    degradation_metric STRING COMMENT 'Name of the metric being tracked (e.g., "vibration", "efficiency", "error_rate")',
    trend_values STRING COMMENT 'Comma-separated time-ordered values (oldest to newest, e.g., "2.1,2.4,2.8,3.1,3.5")',
    alarm_threshold DOUBLE COMMENT 'The threshold value that triggers an alarm or required action',
    time_interval_days INT DEFAULT 7 COMMENT 'Days between each measurement in trend_values',
    action_threshold DOUBLE DEFAULT 0 COMMENT 'Optional earlier threshold for planned action (0 = same as alarm)',
    units STRING DEFAULT '' COMMENT 'Unit of measurement (e.g., "mm/s", "percent", "ppm")'
)
RETURNS STRING
COMMENT 'Predicts time to failure or required action based on degradation trends. Analyzes trend direction, acceleration, and projects when alarm/action thresholds will be reached. Returns JSON with: estimated_days_to_alarm, estimated_days_to_action, degradation_rate, trend_classification, confidence, optimal_intervention_window, risk_if_deferred.'
RETURN ai_query(
    'databricks-claude-sonnet-4',
    CONCAT(
        'You are a {{domain}} predictive analytics expert. Analyze this degradation trend and forecast time to threshold.\n\n',
        'Entity: ', entity_id, ' (', entity_type, ')\n',
        'Metric: ', degradation_metric, CASE WHEN units != '' THEN CONCAT(' (', units, ')') ELSE '' END, '\n',
        'Trend values (oldest→newest): ', trend_values, '\n',
        'Measurement interval: ', CAST(time_interval_days AS STRING), ' days\n',
        'Alarm threshold: ', CAST(alarm_threshold AS STRING), '\n',
        CASE WHEN action_threshold > 0 THEN CONCAT('Action threshold: ', CAST(action_threshold AS STRING), '\n') ELSE '' END,
        '\nAnalyze: trend direction, rate of change, acceleration/deceleration, and project when thresholds will be reached.\n',
        '\nReturn ONLY valid JSON with fields: estimated_days_to_alarm (number), estimated_days_to_action (number), ',
        'degradation_rate (string), trend_classification (LINEAR/ACCELERATING/DECELERATING/STABLE), ',
        'confidence (HIGH/MEDIUM/LOW), optimal_intervention_window (string), risk_if_deferred (string).\n',
        'Output raw JSON starting with { and ending with }.'
    ),
    modelParameters => named_struct('max_tokens', 400, 'temperature', 0.2)
);
*/

-- ─────────────────────────────────────────────────────────────────────────────
-- HOW TO USE THESE TEMPLATES:
-- ─────────────────────────────────────────────────────────────────────────────
-- 1. Uncomment ONE template at a time
-- 2. Replace {{catalog}}.{{schema}} with your actual catalog.schema
-- 3. Replace {{domain}} with your domain name in the prompt
-- 4. Customize the function name, parameters, and prompt for your use case
-- 5. Run the cell to create the function
-- 6. Repeat for each function you need (aim for 5-10)
--
-- TIP: After creating functions, verify them with the next cell.
-- ─────────────────────────────────────────────────────────────────────────────

SELECT 'Templates ready - uncomment and customize one at a time' AS status;

In [0]:
# =============================================================================
# VERIFY UC FUNCTIONS ARE MCP-READY
# =============================================================================
# Lists all functions in your schema and checks they have proper metadata.
# Functions without COMMENTs will appear as unnamed tools in MCP clients.
# =============================================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print(f"Functions in {CATALOG}.{SCHEMA}:")
print("=" * 70)

functions = list(w.functions.list(catalog_name=CATALOG, schema_name=SCHEMA))

if not functions:
    print(f"  No functions found in {CATALOG}.{SCHEMA}")
    print(f"  Create functions using the templates in the previous cell.")
else:
    for func in functions:
        has_comment = bool(func.comment)
        param_count = len(func.input_params.parameters) if func.input_params else 0
        params_with_comments = sum(
            1 for p in (func.input_params.parameters if func.input_params else [])
            if p.comment
        )
        status = "✅" if has_comment and params_with_comments == param_count else "⚠️"
        print(f"  {status} {func.name}")
        print(f"     Comment: {func.comment[:80] + '...' if func.comment and len(func.comment) > 80 else func.comment or 'MISSING ← Add a COMMENT!'}")
        print(f"     Parameters: {param_count} ({params_with_comments} with comments)")
        print()

    print("=" * 70)
    print(f"Total: {len(functions)} function(s)")
    print()
    print(f"MCP Endpoint: {FUNCTIONS_MCP_URL}")
    print(f"Full URL:     https://{{workspace}}{FUNCTIONS_MCP_URL}")
    print()
    if all(f.comment for f in functions):
        print("✅ All functions have COMMENTs — MCP-ready!")
    else:
        print("⚠️  Some functions are missing COMMENTs. Add them for better MCP tool descriptions.")

# Step 5: Create Genie Space

## What Is the Genie Space MCP Server?

The Genie Space MCP server lets the agent query your data tables using **natural language** — no SQL needed from the user. The agent sends a question like *"How many events occurred last month?"* and Genie translates it to SQL, runs it, and returns results.

```
/api/2.0/mcp/genie/{space_id}
```

## How to Create a Genie Space

You can create one via:
1. **UI** — Navigate to the Genie section in your workspace, click "New", add your tables
2. **API/SDK** — Use the code in the next cell

## What Tables to Include

Include all tables the agent might need to query:
- **Fact tables** — Events, transactions, measurements, time series
- **Dimension tables** — Equipment master, customer master, product catalog
- **Avoid** — Staging tables, temporary tables, raw unprocessed data

## Sample Questions

After creating the space, add 5-10 sample questions in the Genie UI. These improve accuracy:
- Questions that match how your users actually ask
- Questions that exercise different tables and join patterns
- Questions with time filters ("last month", "this year")
- Questions with aggregations ("total", "average", "count")

## Getting the Space ID

After creation, the space ID appears in the URL:
```
https://your-workspace.databricks.com/genie/rooms/{space_id}
```

Copy this ID back into the `GENIE_SPACE_ID` variable in Cell 2.

---

> **💡 Assistant Shortcut:** You can ask the Databricks Assistant to create the Genie space for you:
>
> *"Create a Genie space called 'My Domain Agent' with tables my_catalog.my_schema.events and my_catalog.my_schema.equipment. Add sample questions about event counts, equipment health trends, and regional comparisons."*
>
> The assistant will create the space, add your tables, and return the space ID for you to paste into the configuration cell.

In [0]:
# =============================================================================
# CREATE GENIE SPACE
# =============================================================================
# Creates a Genie space with your domain tables.
# After creation, copy the space_id back to the GENIE_SPACE_ID config variable.
# =============================================================================

# NOTE: The Databricks SDK Genie API may vary by workspace version.
# If this code doesn't work, create the space via the UI and paste the ID below.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
WORKSPACE_HOST = spark.conf.get("spark.databricks.workspaceUrl")

if GENIE_SPACE_ID:
    print(f"Genie space already configured: {GENIE_SPACE_ID}")
    print(f"  URL: https://{WORKSPACE_HOST}/genie/rooms/{GENIE_SPACE_ID}")
    print(f"  MCP: /api/2.0/mcp/genie/{GENIE_SPACE_ID}")
else:
    print(f"""
{'='*70}
CREATE YOUR GENIE SPACE
{'='*70}

Option 1: Create via UI (Recommended)
  1. Go to: https://{WORKSPACE_HOST}/genie
  2. Click "New" to create a new space
  3. Name it: "{GENIE_SPACE_NAME}"
  4. Add these tables:
""")
    for table in DOMAIN_TABLES:
        print(f"     - {table}")
    
    print(f"""
  5. Add 5-10 sample questions relevant to your domain
  6. Save and copy the space_id from the URL
  7. Paste it into GENIE_SPACE_ID in Cell 2

Option 2: After creating, set the variable here:
""")
    print(f'  GENIE_SPACE_ID = "<paste-your-space-id-here>"')
    print(f"""  
  Then re-run Cell 2 to update derived URLs.

{'='*70}
Genie MCP URL (after creation):
  /api/2.0/mcp/genie/<space_id>
{'='*70}
""")

# Step 6: Generate Reference Documents

## Why Reference Documents?

The AI Search MCP server provides the agent with **domain knowledge** that isn't in your data tables:
- Industry standards and specifications
- Best practices and procedures
- Diagnostic decision trees
- Glossaries and terminology
- Regulatory requirements

This is the knowledge a domain expert carries in their head. Without it, the agent can query data but can't *interpret* it against standards.

## Options for Creating Documents

| Option | Best For | Effort |
|--------|---------|--------|
| **Upload existing PDFs** | You already have SOPs, standards, manuals | Low — just copy to volume |
| **Write markdown manually** | You need precise, curated content | Medium — write 3-5 docs |
| **Generate with AI** | Bootstrapping / getting started quickly | Low — run the next cell |
| **Combination** | Production use | Medium — generate drafts, then refine |

## Recommended Documents (3-5)

For most domains, create documents covering:

1. **Diagnostics & Troubleshooting** — How to diagnose common issues in your domain
2. **Standards & Thresholds** — What values are normal, concerning, critical
3. **Operations & Processes** — How things work, process flows, key concepts
4. **Safety & Compliance** — Regulations, safety procedures, compliance requirements
5. *(Optional)* **Instrumentation & Data** — What sensors/metrics mean, data quality

## Volume Setup

Documents go into a Unity Catalog Volume:
```
/Volumes/{catalog}/{schema}/reference_docs/
```

Supported formats:
- **Markdown (.md)** — Chunked by `## ` section headers (best for structured content)
- **PDF (.pdf)** — Parsed with `ai_parse_document()`, then chunked by paragraphs

---

> **💡 Assistant Shortcut:** You can ask the Databricks Assistant to generate your reference documents:
>
> *"Generate 4 reference documents about pharmaceutical manufacturing: one on GMP compliance, one on deviation investigation procedures, one on equipment qualification, and one on process validation. Save them as markdown files in /Volumes/my_catalog/my_schema/reference_docs/"*
>
> The assistant will craft domain-appropriate content using `ai_query()`, save the files to your volume, and confirm the results. You can also ask it to upload existing PDFs or refine generated content.

In [0]:
# =============================================================================
# GENERATE REFERENCE DOCUMENTS WITH AI
# =============================================================================
# Uses ai_query() to generate domain-specific reference documents.
# Customize the prompts below for your domain.
# =============================================================================

import time

# --- Document Definitions ---
# Customize these for your domain. Each entry generates one markdown document.
# The 'topic' and 'guidance' fields are sent to the AI model.

DOCUMENTS = [
    {
        "filename": "diagnostics_and_troubleshooting.md",
        "topic": "Diagnostics and Troubleshooting",
        "guidance": f"""Write a comprehensive reference document for {DOMAIN_NAME} diagnostics.
            Cover: common failure modes, diagnostic decision trees, symptom-to-cause mapping,
            severity classification, and recommended investigation procedures.
            Include specific thresholds, metrics, and domain terminology.
            Format as markdown with ## section headers."""
    },
    {
        "filename": "standards_and_thresholds.md",
        "topic": "Standards and Thresholds",
        "guidance": f"""Write a reference document covering key standards, thresholds, and limits for {DOMAIN_NAME}.
            Include: industry standards (ISO, API, ASTM as applicable), normal operating ranges,
            warning thresholds, critical limits, and how to interpret values against these benchmarks.
            Format as markdown with ## section headers."""
    },
    {
        "filename": "operations_and_processes.md",
        "topic": "Operations and Processes",
        "guidance": f"""Write a reference document explaining key operational processes in {DOMAIN_NAME}.
            Cover: core workflows, process descriptions, operational modes, key concepts
            and terminology, and how different subsystems interact.
            Format as markdown with ## section headers."""
    },
    {
        "filename": "safety_and_compliance.md",
        "topic": "Safety and Compliance",
        "guidance": f"""Write a reference document on safety procedures and compliance requirements for {DOMAIN_NAME}.
            Cover: safety classification systems, incident severity tiers, reporting requirements,
            emergency procedures, and regulatory frameworks.
            Format as markdown with ## section headers."""
    },
]

# --- Generation ---
def generate_and_save_doc(filename, topic, guidance):
    """Generate a document using ai_query and save to the reference volume."""
    print(f"  Generating: {filename}...", end=" ", flush=True)
    start = time.time()
    
    prompt = f"""{guidance}

Domain context: {DOMAIN_DESCRIPTION}

Write approximately 3000-4000 words. Be specific and technical.
Start directly with a # title, then use ## for major sections."""
    
    result = spark.sql(
        "SELECT ai_query(:model, :prompt, modelParameters => named_struct('temperature', CAST(0.3 AS DOUBLE), 'max_tokens', CAST(4096 AS INT))) AS content",
        args={"model": AI_MODEL, "prompt": prompt}
    ).collect()[0]["content"]
    
    # Save to volume
    filepath = f"{VOLUME_PATH}/{filename}"
    dbutils.fs.put(filepath, result, overwrite=True)
    elapsed = time.time() - start
    size_kb = len(result.encode()) / 1024
    print(f"\u2705 ({size_kb:.1f} KB, {elapsed:.0f}s)")
    return result

# --- Execute ---
print(f"{'='*70}")
print(f"GENERATING REFERENCE DOCUMENTS FOR: {DOMAIN_NAME}")
print(f"Volume: {VOLUME_PATH}")
print(f"{'='*70}")

for doc in DOCUMENTS:
    generate_and_save_doc(doc["filename"], doc["topic"], doc["guidance"])

print(f"\n{'='*70}")
print(f"COMPLETE — {len(DOCUMENTS)} documents generated")
print(f"{'='*70}")
print(f"\nFiles in volume:")
for f in dbutils.fs.ls(VOLUME_PATH):
    print(f"  {f.name} ({f.size / 1024:.1f} KB)")
print(f"\nNext: Run the indexing cell to create the Vector Search index.")

# Step 7: Create AI Search Index

## How AI Search MCP Works

The AI Search MCP server performs **semantic retrieval** over your reference documents:

1. Documents are split into chunks (~1500 characters each)
2. Each chunk is embedded into a vector using a language model
3. At query time, the user's question is embedded and compared against chunks
4. The most relevant chunks are returned to the agent

```
/api/2.0/mcp/ai-search/{catalog}/{schema}
```

## Chunking Strategy

| File Type | Strategy | How It Works |
|-----------|----------|-------------|
| Markdown (.md) | Section-based | Split on `## ` headers, then by paragraphs if sections are too large |
| PDF (.pdf) | Paragraph-based | Extract text with `ai_parse_document()`, split on line breaks, then sentences |

Chunks include 200-character overlap for context continuity between adjacent chunks.

## Index Configuration

| Setting | Value | Why |
|---------|-------|-----|
| Embedding model | `databricks-gte-large-en` | Good balance of quality and speed |
| Max chunk size | 1500 chars | Fits within embedding context window |
| Pipeline type | TRIGGERED | Syncs on-demand (not continuous) |
| Change Data Feed | Enabled | Required for Delta Sync indexes |

## The Pipeline

```
Volume (docs) → Chunk → Delta Table → Vector Search Index → AI Search MCP
```

---

> **💡 Assistant Shortcut:** You can ask the Databricks Assistant to handle the entire indexing pipeline:
>
> *"Index the reference documents in /Volumes/my_catalog/my_schema/reference_docs/ into a Vector Search index. Use the endpoint 'my-ai-search-endpoint' and the databricks-gte-large-en embedding model. Support both markdown and PDF files."*
>
> The assistant will chunk your documents (using section-based splitting for markdown and paragraph-based splitting for PDFs), create the Delta table with CDC enabled, and set up the Vector Search index — handling any errors along the way.

In [0]:
# =============================================================================
# INDEX DOCUMENTS INTO VECTOR SEARCH
# =============================================================================
# Reads all .md and .pdf files from the volume, chunks them, writes to a
# Delta table, and creates/syncs a Vector Search index.
# =============================================================================

import uuid
import re
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
    VectorIndexType,
)

w = WorkspaceClient()

# --- Step 1: List documents in volume ---
print(f"Documents in volume ({VOLUME_PATH}):")
all_files = dbutils.fs.ls(VOLUME_PATH)
md_files = [f for f in all_files if f.name.endswith(".md")]
pdf_files = [f for f in all_files if f.name.endswith(".pdf")]
for f in all_files:
    print(f"  {f.name} ({f.size / 1024:.1f} KB)")
print(f"\nFound {len(md_files)} markdown + {len(pdf_files)} PDF files")
assert md_files or pdf_files, f"No .md or .pdf files found in {VOLUME_PATH}."

# --- Step 2: Chunking functions ---
def chunk_markdown(text, max_chars=MAX_CHUNK_CHARS):
    """Split markdown by ## sections, then by paragraphs if too large."""
    chunks = []
    sections = re.split(r'(?=^## )', text, flags=re.MULTILINE)
    for section in sections:
        section = section.strip()
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
        else:
            paragraphs = section.split('\n\n')
            current_chunk = ""
            for para in paragraphs:
                if len(current_chunk) + len(para) + 2 > max_chars and current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = para
                else:
                    current_chunk += ("\n\n" if current_chunk else "") + para
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
    return chunks

def chunk_pdf_text(text, max_chars=MAX_CHUNK_CHARS, overlap_chars=200):
    """Multi-strategy chunker for PDF text: paragraphs > lines > sentences > hard split."""
    if text.count('\n\n') > 5:
        segments = [s.strip() for s in text.split('\n\n') if s.strip()]
    elif text.count('\n') > 10:
        segments = [s.strip() for s in text.split('\n') if s.strip()]
    else:
        segments = re.split(r'(?<=\.)\s+(?=[A-Z])', text)
        segments = [s.strip() for s in segments if s.strip()]

    # Force-split segments that are still too large
    fine_segments = []
    for seg in segments:
        if len(seg) <= max_chars:
            fine_segments.append(seg)
        else:
            sentences = re.split(r'(?<=[\.!\?])\s+', seg)
            for sent in sentences:
                if len(sent) <= max_chars:
                    fine_segments.append(sent)
                else:
                    words = sent.split()
                    part = ""
                    for word in words:
                        if len(part) + len(word) + 1 > max_chars and part:
                            fine_segments.append(part.strip())
                            part = word
                        else:
                            part += (" " if part else "") + word
                    if part.strip():
                        fine_segments.append(part.strip())

    # Accumulate segments into chunks with overlap
    chunks = []
    current_chunk = ""
    for seg in fine_segments:
        if len(current_chunk) + len(seg) + 2 > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            if overlap_chars > 0 and len(current_chunk) > overlap_chars:
                current_chunk = current_chunk[-overlap_chars:] + "\n\n" + seg
            else:
                current_chunk = seg
        else:
            current_chunk += ("\n\n" if current_chunk else "") + seg
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# --- Step 3: Process all files ---
all_chunks = []

if md_files:
    print("\nProcessing markdown files...")
    for f in md_files:
        content = dbutils.fs.head(f.path, 200000)
        doc_chunks = chunk_markdown(content)
        for i, chunk_text in enumerate(doc_chunks):
            all_chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "chunk_position": i,
                "chunk_to_embed": chunk_text,
                "chunk_to_retrieve": chunk_text,
                "source_uri": f.path,
                "document_name": f.name,
            })
        print(f"  {f.name}: {len(doc_chunks)} chunks")

if pdf_files:
    print("\nProcessing PDF files...")
    for f in pdf_files:
        pdf_path = f.path.replace("dbfs:", "")
        pdf_text_df = spark.sql(f"SELECT ai_parse_document('{pdf_path}') AS parsed")
        parsed_result = pdf_text_df.collect()[0]["parsed"]
        if hasattr(parsed_result, 'parsed_text'):
            pdf_text = parsed_result.parsed_text
        elif hasattr(parsed_result, 'text'):
            pdf_text = parsed_result.text
        elif isinstance(parsed_result, dict):
            pdf_text = parsed_result.get('parsed_text', parsed_result.get('text', str(parsed_result)))
        else:
            pdf_text = str(parsed_result)

        doc_chunks = chunk_pdf_text(pdf_text)
        for i, chunk_text in enumerate(doc_chunks):
            all_chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "chunk_position": i,
                "chunk_to_embed": chunk_text,
                "chunk_to_retrieve": chunk_text,
                "source_uri": f.path,
                "document_name": f.name,
            })
        print(f"  {f.name}: {len(doc_chunks)} chunks (extracted {len(pdf_text)} chars)")

print(f"\nTotal chunks: {len(all_chunks)}")

# --- Step 4: Write chunks to Delta table ---
chunks_df = spark.createDataFrame(all_chunks)
chunks_df.write.format("delta").mode("overwrite").saveAsTable(CHUNKS_TABLE)
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"\nChunks per document:")
display(spark.sql(f"SELECT document_name, COUNT(*) AS chunks, ROUND(AVG(LENGTH(chunk_to_embed))) AS avg_chunk_len FROM {CHUNKS_TABLE} GROUP BY document_name ORDER BY document_name"))

# --- Step 5: Create or sync Vector Search index ---
try:
    w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=AI_SEARCH_ENDPOINT,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=CHUNKS_TABLE,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="chunk_to_embed",
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
            columns_to_sync=["chunk_id", "chunk_to_embed", "chunk_to_retrieve",
                             "source_uri", "chunk_position", "document_name"],
        ),
    )
    print(f"\n\u2705 Index created: {INDEX_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
        print(f"\n\u2705 Index already exists — triggered sync: {INDEX_NAME}")
    else:
        raise

print(f"  AI Search MCP URL: {AI_SEARCH_MCP_URL}")
print(f"  Endpoint: {AI_SEARCH_ENDPOINT}")
print(f"  Embedding: {EMBEDDING_MODEL}")

# Step 8: Grant Permissions to Service Principal

## Why Permissions?

External clients (like the Spotfire Agent) authenticate as a **service principal** via OAuth M2M. That service principal needs explicit Unity Catalog grants to access your MCP servers.

## Required Grants

| Grant | Purpose | Required For |
|-------|---------|-------------|
| `USE CATALOG` | Navigate to the catalog | All MCP servers |
| `USE SCHEMA` | Navigate to the schema | All MCP servers |
| `EXECUTE` | Call UC Functions | UC Functions MCP |
| `SELECT` | Read tables | Genie MCP + SQL MCP |
| `READ VOLUME` | Access reference docs | AI Search MCP |
| `CAN_RUN` (Genie space) | Execute Genie queries | Genie MCP |

## Service Principal Setup

If you don't have a service principal yet:
1. Go to **Account Console → Service Principals → Add**
2. Create a new SP (e.g., "my_agent_service")
3. Generate a client secret
4. Note the `client_id` (UUID) — this is your `SERVICE_PRINCIPAL_ID`
5. Store the `client_secret` securely (you'll need it for OAuth)

In [0]:
%sql
-- =============================================================================
-- GRANT PERMISSIONS TO SERVICE PRINCIPAL
-- =============================================================================
-- Run this cell to grant the necessary permissions.
-- Replace the service principal ID if you haven't set it in the config cell.
-- =============================================================================

-- 1. Catalog navigation
GRANT USE CATALOG ON CATALOG ${CATALOG} TO `${SERVICE_PRINCIPAL_ID}`;

-- 2. Schema navigation
GRANT USE SCHEMA ON SCHEMA ${CATALOG}.${SCHEMA} TO `${SERVICE_PRINCIPAL_ID}`;

-- 3. Execute functions (UC Functions MCP server)
GRANT EXECUTE ON SCHEMA ${CATALOG}.${SCHEMA} TO `${SERVICE_PRINCIPAL_ID}`;

-- 4. Read tables (Genie + SQL MCP servers)
GRANT SELECT ON SCHEMA ${CATALOG}.${SCHEMA} TO `${SERVICE_PRINCIPAL_ID}`;

-- 5. Read volume (AI Search - reference docs)
GRANT READ VOLUME ON SCHEMA ${CATALOG}.${SCHEMA} TO `${SERVICE_PRINCIPAL_ID}`;

In [0]:
# =============================================================================
# GRANT CAN_RUN ON GENIE SPACE
# =============================================================================
# The Genie space needs a separate permission grant (not UC-based).
# =============================================================================

import requests

if not GENIE_SPACE_ID:
    print("⚠️  GENIE_SPACE_ID not set. Create the Genie space first (Step 5),")
    print("   then set the ID in Cell 3 and re-run.")
else:
    WORKSPACE_HOST = spark.conf.get("spark.databricks.workspaceUrl")
    
    # Get notebook token for API call
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    
    resp = requests.patch(
        f"https://{WORKSPACE_HOST}/api/2.0/permissions/genieSpaces/{GENIE_SPACE_ID}",
        headers={"Authorization": f"Bearer {token}"},
        json={"access_control_list": [{
            "service_principal_name": SERVICE_PRINCIPAL_ID,
            "all_permissions": [{"permission_level": "CAN_RUN"}]
        }]}
    )
    
    if resp.status_code == 200:
        print(f"✅ Granted CAN_RUN on Genie space {GENIE_SPACE_ID}")
        print(f"   Service Principal: {SERVICE_PRINCIPAL_ID}")
    else:
        print(f"❌ Failed: {resp.status_code}")
        print(f"   Response: {resp.text[:200]}")
        print(f"\n   Manual alternative: Open the Genie space UI and add the SP with CAN_RUN.")
        print(f"   URL: https://{WORKSPACE_HOST}/genie/rooms/{GENIE_SPACE_ID}")

# Step 9: Behavior Pack — Authoring Guide

## What Is a Behavior Pack?

The Spotfire Agent uses a **behavior pack** — a folder of plain-text files — that transforms the generic Databricks Agent into your domain expert. **No code required** — domain experts can author these files directly.

The pack contains:
- **`pack.yaml`** — manifest (agent card: name, description, version, skill list)
- **`system_prompt.md`** — persona + routing rules (decision order, guardrails)
- **`AGENTS.md`** — durable domain knowledge (terms, thresholds, composition patterns)
- **`help.md`** — verbatim text shown when user types "help" (not model-generated)
- **`skills/`** — one SKILL.md per capability (4 folders)

## SKILL.md Format

Each skill has **YAML frontmatter** + **Markdown instructions**:

```markdown
---
name: genie-data-questions            # MUST equal the folder name
description: >-                        # THIS is the routing surface (1–1024 chars, keyword-rich!)
  Retrieve operational data from the scoped Genie space using natural language.
  PRIMARY path for any question needing rows — events, trends, "which..."
allowed-tools: query_space_<ID> poll_response_<ID>   # SPACE-separated!
---

# Genie — My Domain Data

## What it is
A scoped Databricks Genie space over the domain tables. The primary data path.

## Tools
- `query_space_<SPACE_ID>` — submit a natural-language question
- `poll_response_<SPACE_ID>` — poll an in-progress query

## When to use
- "What events occurred on [entity] last week?"
- Any question needing rows from the database.

## Workflow
1. Ask with query_space_...; resolve vague windows into dates.
2. Poll if in-progress.
3. Relay result faithfully.

## Fallback
On error → fall back to sql-execution.

## Output
All rows, no ellipsis. Never fabricate data.
```

**Recommended body sections** (in order): What it is → Tools → When to use → Workflow → Fallback → Output.

## Critical Rules

| Rule | Detail |
|------|--------|
| Folder name = `name:` field | If they differ, the skill is **silently skipped** |
| Naming convention | Lowercase letters, digits, hyphens only (1–64 chars) |
| `allowed-tools` format | **Space-separated** (commas will break parsing!) |
| Progressive disclosure | Only `name` + `description` loaded upfront; body loaded on-demand |
| `description` is key | Must include domain keywords and trigger intents (1–1024 chars) |
| Extra files in skill folders | Allowed (e.g. a reference `.md`), but only `SKILL.md` is parsed as the skill definition |

## pack.yaml Fields

| Field | Required | Notes |
|-------|----------|-------|
| `name` | ✔ | Shown in the agent picker (defaults to "Databricks Agent") |
| `description` | ✔ | One paragraph: what the agent does and for whom |
| `version` | ✔ | Any string (e.g. `0.1.0`). Bump when you change the pack |
| `skills[].id` | ✔ | Must match a `skills/<id>/` folder |
| `skills[].name` | ✔ | Human-readable label |
| `skills[].description` | ✔ | Short blurb |
| `skills[].tags` | optional | Free-form keywords |

> **`help-and-capabilities`** is a special skill handled by the agent's **built-in middleware** using your `help.md`. Advertise it in `pack.yaml` (so it appears on the agent card) but do **NOT** create a `skills/help-and-capabilities/` folder — it will be silently ignored.

## help.md

Write `help.md` as a friendly capability summary with **concrete starter prompts grouped by capability** (data, computation, documents, discovery). The text is returned **verbatim** when a user types "help" — the model is not invoked.

## Tool Name Patterns

Authors need these to fill in `allowed-tools` correctly:

| Capability | Tool Name Pattern | Example |
|-----------|-------------------|--------|
| Genie | `query_space_<SPACE_ID>` `poll_response_<SPACE_ID>` | `query_space_01f19cc5…` |
| UC Functions | `<catalog>__<schema>__<function>` | `my_catalog__my_schema__diagnose_issue` |
| AI Search | `<catalog>__<schema>__<index>` | `my_catalog__my_schema__reference_index` |
| SQL | `execute_sql` `execute_sql_read_only` `poll_sql_result` | (fixed names, always the same) |

> Dots in catalog/schema/function names become **double underscores** in tool names.
> If you don't yet know the space id, use a placeholder like `query_space_<SPACE_ID>` and tell the operator to replace it at deploy time.

## Who Authors What

| File | Primary Author | Content Focus |
|------|---------------|---------------|
| `pack.yaml` | Product owner | Public identity, skill inventory |
| `system_prompt.md` | Domain expert + data engineer | Routing logic, guardrails, output policies |
| `AGENTS.md` | Domain expert | Knowledge, thresholds, formulas, patterns |
| `help.md` | Product owner | User-facing starter prompts |
| `skills/genie-data-questions/` | Data engineer | Tables, tool names, query workflow |
| `skills/uc-functions/` | Domain expert + data engineer | Function list, retrieve-then-compute patterns |
| `skills/vector-search-retrieval/` | Data engineer | Index name, grounding rules |
| `skills/sql-execution/` | Data engineer | Generic (can copy between domains) |

---

## Best Practices (Lessons from Production Agents)

These recommendations come from building 3 production domain agents (Petroleum, Semiconductor, Energy Operations).

### 1. Make `uc-functions` Description Exhaustive

The `description` is the **routing trigger** — it's the only thing loaded upfront. A vague description gets skipped.

**Bad:**
```yaml
description: Domain diagnostics and assessments.
```

**Good:**
```yaml
description: >-
  Equipment event root-cause analysis (diagnose_equipment_event), compressor
  health scoring from vibration/pressure/temperature (assess_compressor_health),
  operational mode classification, sensor anomaly interpretation, predictive
  degradation forecasting, SPC control chart evaluation, trip severity assessment,
  event correlation, cross-plant benchmarking, and maintenance recommendations.
  Use when the user asks "why", "what caused", "how healthy", "when will it fail",
  "compare", "should we quarantine", or needs expert interpretation of data.
```

Include: every function name, every trigger verb, every domain noun users will actually say.

### 2. Keep `system_prompt.md` Short (≈40 lines)

Only include:
- Identity (2 sentences)
- The 4 capabilities (1 bullet each, naming your specific resources)
- Routing decision order (5 numbered rules)
- Guardrails (never invent data, read-only default, retrieve before compute)
- Table-output policy (no ellipsis, relay all rows)
- A pointer: "See AGENTS.md for domain knowledge and patterns."

Do NOT put domain facts, thresholds, or composition workflows here.

### 3. Put Everything Domain-Specific in `AGENTS.md`

`AGENTS.md` is the agent's "memory" — it can be long (≈150 lines). Include:
- Full domain knowledge table (thresholds, formulas, enumerations)
- ALL composition patterns (Genie→Function→Synthesize) with worked examples
- Response guidelines specific to your domain
- Error handling and retry logic
- Entity naming conventions and data quirks

### 4. Include Table Schemas in `genie-data-questions` Body

Helps the agent formulate better Genie questions:
```markdown
## Tables
- `plants_event_frames` — EVENT_TYPE (17 types), SEVERITY (Low/Med/High/Critical),
  EQUIPMENT_KEY, DURATION_HR, IS_TRIP, START_DATETIME, END_DATETIME
- `plants_timeseries` — TAG, VALUE, UNITS, DATETIME (1-min resolution, 6 years)
```

### 5. Copy `sql-execution` Verbatim Between Domains

It's always the same 3 tool names (`execute_sql`, `execute_sql_read_only`, `poll_sql_result`) and the same workflow. No customization needed.

### 6. Consider a 5th Skill for Complex Workflows (Optional)

For domains with many functions and multi-step patterns, add:
```
skills/multi-tool-workflows/SKILL.md
```

With description:
```yaml
description: >-
  Orchestrate multi-step retrieve-then-compute workflows: event diagnosis
  (Genie→diagnose_equipment_event), health checks (Genie→assess_compressor_health),
  predictive maintenance (Genie→forecast_degradation). Use when the question
  requires BOTH data retrieval AND expert interpretation in sequence.
```

This gives the agent an explicit routing target for composition rather than relying on implicit chaining.

### 7. Recommended Content Distribution

| File | Content | Target Length |
|------|---------|---------------|
| `system_prompt.md` | Identity + routing + guardrails + table policy | ≈40 lines |
| `AGENTS.md` | Domain knowledge + ALL composition patterns + response rules + error handling | ≈150 lines |
| `genie-data-questions/SKILL.md` | Table schemas + tool names + query workflow + fallback | ≈40 lines |
| `uc-functions/SKILL.md` | All function names + signatures + retrieve-then-compute | ≈60 lines |
| `vector-search/SKILL.md` | Index name + document list + grounding rules | ≈30 lines |
| `sql-execution/SKILL.md` | Generic (copy between domains) | ≈30 lines |

### 8. Common Mistakes That Break Routing

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Vague `description` in uc-functions | Agent never calls your functions | Add every function name + trigger verb |
| Composition patterns in system_prompt | System prompt too long, degrades quality | Move to AGENTS.md |
| Commas in `allowed-tools` | Tools not recognized | Use spaces: `tool_a tool_b tool_c` |
| Generic keywords in description | Wrong skill triggered | Use domain-specific nouns ("vibration", "yield", "stockout") |
| Folder name ≠ SKILL.md `name:` | Skill silently ignored | Make them identical |

---

## 💡 Assistant Shortcut: Let the Databricks Assistant Generate Your Pack

You don't have to write these files from scratch. The Databricks Assistant can generate any or all of the behavior pack files. Here are effective prompts for each:

| File | Prompt |
|------|--------|
| **Full pack** | *"Generate a complete behavior pack for my [domain] with tables [table1, table2], Genie space [ID], and UC functions [func1, func2, ...]. Save it to /Volumes/[catalog]/[schema]/behavior_packs/"* |
| **AGENTS.md** | *"Write an AGENTS.md for my [domain] agent. Include: entity ID format [X], key thresholds [metric1 > Y is critical], composition patterns for diagnose/assess/forecast workflows, and response guidelines."* |
| **system_prompt.md** | *"Write a lean system_prompt.md (~35 lines) for my [domain] agent with routing rules for 4 MCP servers: Genie (data), UC functions (computation), AI Search (documents), SQL (discovery/fallback)."* |
| **SKILL.md (uc-functions)** | *"Write the uc-functions SKILL.md for my schema [catalog.schema] with these functions: [list all]. Make the description exhaustive — include every function name and trigger verb."* |
| **SKILL.md (genie)** | *"Write the genie-data-questions SKILL.md for Genie space [ID] with tables: [table1] (columns: col1, col2, ...), [table2] (columns: ...)."* |
| **help.md** | *"Write a help.md with starter prompts for a [domain] agent that can query [data type], run [function types], and search [document types]."* |

**Tips for best results:**
- Provide your actual function names, table schemas, and Genie space ID
- Ask for the full file content (not just a skeleton)
- Request JSON output format instructions in function descriptions
- After generation, review and customize the domain knowledge in AGENTS.md

> The next cell (Step 9b) also generates the complete pack programmatically — run it and customize, or use the prompts above for more control over individual files.

In [0]:
# =============================================================================
# STEP 9b: GENERATE COMPLETE BEHAVIOR PACK
# =============================================================================
# Creates the full folder structure required by the Spotfire Agent:
#   pack.yaml, system_prompt.md, AGENTS.md, help.md, + 4 skills
# Files are saved to a UC Volume for easy retrieval.
# =============================================================================

import os

# Derive names for the pack
domain_slug = DOMAIN_NAME.lower().replace(' ', '-')
domain_under = DOMAIN_NAME.lower().replace(' ', '_')
space_id_placeholder = GENIE_SPACE_ID if GENIE_SPACE_ID else "<SPACE_ID>"
catalog_schema_tool = f"{CATALOG}__{SCHEMA}"

# --- Auto-discover UC functions (if schema already has functions) ---
# If you've already run Steps 3-4, this picks up your real function names.
# Otherwise, falls back to placeholders you'll replace in the next-steps.
try:
    _fn_rows = spark.sql(f"SHOW USER FUNCTIONS IN {CATALOG}.{SCHEMA}").collect()
    discovered_functions = [
        row[0].split(".")[-1] for row in _fn_rows
        if not row[0].startswith("spark_catalog")
    ]
    if discovered_functions:
        print(f"\u2705 Auto-discovered {len(discovered_functions)} UC functions: {', '.join(discovered_functions)}")
    else:
        discovered_functions = None
except Exception:
    discovered_functions = None
    print("⚠️  Could not list functions (schema may not exist yet). Using placeholders.")
    print("   After creating functions (Steps 3-4), re-run this cell to auto-populate.")

# --- pack.yaml ---
pack_yaml = f"""name: "{DOMAIN_NAME} Agent"
description: >-
  A {DOMAIN_NAME.lower()} assistant on Databricks. Answers operational data
  questions via Genie, runs domain diagnostics and assessments with Unity
  Catalog functions, searches reference documents via AI Search, and
  discovers the catalog with SQL.
version: "1.0.0"

skills:
  - id: genie-data-questions
    name: Genie data questions
    description: >-
      Retrieve {DOMAIN_NAME.lower()} operational data from the scoped Genie
      space using natural language. PRIMARY path for any question needing rows
      from the database — events, measurements, time series, comparisons,
      "which entities...", "show me...", "what happened...". Genie writes and
      runs SQL for you.
    tags: [databricks, genie, data, {domain_slug}]
  - id: uc-functions
    name: Unity Catalog functions
    description: >-
      Domain diagnostics, health assessments, forecasting, and recommendations.
      Includes all UC functions in {CATALOG}.{SCHEMA}. Use when the user asks
      "why", "what caused", "how healthy", "when will it fail", "compare",
      "should we quarantine/ship/hold", or needs expert interpretation of
      retrieved data. {DOMAIN_NAME}-specific computations beyond SQL.
    tags: [databricks, unity-catalog, functions, {domain_slug}]
  - id: vector-search-retrieval
    name: AI Search retrieval
    description: >-
      Semantic search over {DOMAIN_NAME.lower()} reference documents and standards.
    tags: [databricks, vector-search, rag, {domain_slug}]
  - id: sql-execution
    name: SQL discovery & fallback
    description: >-
      Catalog/schema/table discovery and a complex-SQL fallback; read-only by
      default.
    tags: [databricks, sql, discovery]
  - id: help-and-capabilities
    name: Help and capabilities
    description: >-
      Onboarding and 'what can you do' — returns starter prompts.
    tags: [help, onboarding]
"""

# --- system_prompt.md ---
# BEST PRACTICE: Keep system_prompt short (~40 lines).
# Identity + capabilities + routing + guardrails + table policy.
# All domain knowledge and composition patterns go in AGENTS.md.
system_prompt_md = f"""You are a {DOMAIN_NAME.lower()} assistant with access to 4 Databricks MCP servers.

## Capabilities
- **Genie** — scoped Genie space over your operational data tables. PRIMARY data path.
- **UC functions** — domain computations: diagnostics, assessments, forecasting.
- **AI Search** — semantic search over reference documents and standards.
- **SQL** — catalog discovery (SHOW/DESCRIBE) + complex-SQL fallback.

## Routing (decision order)
1. Metadata / discovery (tables, columns, functions) → **SQL**.
2. Data from the database (events, measurements, "which...") → **Genie first**;
   also wants interpretation? → pass values to a **UC function**.
3. Best practices / standards / procedures → **AI Search**.
4. Computation / diagnosis / forecast:
   - values provided → **UC function directly**;
   - no values → **Genie** to retrieve, then **UC function**.
5. Genie failed or complex SQL needed → fall back to **SQL**.

## Guardrails
- Prefer Genie over hand-written SQL for data retrieval.
- Never call a function with guessed values — retrieve real data first.
- Never invent tool outputs, function results, passages, or table/column names.
- Default to read-only; confirm before any data-modifying SQL.

## Table output policy
- Relay ALL rows returned. No ellipsis, no placeholder rows.
- Never drop a row because a value is zero, null, or missing.
- Truncation notes go AFTER the table, not inside it.

See **AGENTS.md** for domain knowledge, composition patterns, and response rules.
"""

# --- AGENTS.md ---
# BEST PRACTICE: This is the agent's "memory" — it can be long (~150 lines).
# Put ALL domain knowledge, composition patterns, response rules, and error
# handling here. system_prompt.md just says "See AGENTS.md".
agents_md = f"""# {DOMAIN_NAME} — Agent Domain Knowledge

You help {DOMAIN_NAME.lower()} practitioners work with a Databricks workspace through Genie,
Unity Catalog functions, AI Search, and SQL.

## Capability map
- **Genie** (`DATABRICKS_GENIE_MCP_SERVER_URL`) — operational data tables.
- **UC functions** (`DATABRICKS_FUNCTIONS_MCP_SERVER_URL`) — domain diagnostics and assessments.
- **AI Search** (`DATABRICKS_VECTORSEARCH_MCP_SERVER_URL`) — reference documents.
- **DBSQL** (`DATABRICKS_DBSQL_MCP_SERVER_URL`) — discovery + fallback.

## Domain knowledge

### Entity identification
- **Entity ID format:** `REGION_PLANTTYPE_NN_EQUIPCLASS_ID` (customize for your domain)
- **Regions:** [list your operating regions]
- **Entity types:** [list equipment/asset categories]

### Thresholds and classifications
| Metric | Good | Warning | Critical | Action |
|--------|------|---------|----------|--------|
| [Metric 1] | [range] | [range] | [range] | [action] |
| [Metric 2] | [range] | [range] | [range] | [action] |
| [Metric 3] | [range] | [range] | [range] | [action] |

### Key domain facts
- **Fact 1:** [detail relevant to computations]
- **Fact 2:** [detail relevant to interpreting results]
- **Fact 3:** [formulas, e.g., Water Cut = Water / (Water + Oil) * 100]
- **Normal ranges:** [what values are expected in steady state]
- **Critical limits:** [values requiring immediate intervention]

## Composition patterns (multi-tool workflows)

### Pattern A: Data + Diagnosis
**Triggers:** "What's wrong with...", "Why is... underperforming?", "Diagnose..."
1. **Genie** → Retrieve entity metrics (recent events, sensor readings, history)
2. **UC Function** → Call the appropriate diagnostic function with retrieved values
3. **Synthesize** → Present data + root cause + recommended actions

### Pattern B: Data + Document Context
**Triggers:** "Is this above the standard?", "...compared to best practice?"
1. **Genie** → Get the actual current value(s)
2. **AI Search** → Find the relevant standard/threshold in reference docs
3. **Synthesize** → Compare actual vs. documented guideline, cite source

### Pattern C: Health Assessment
**Triggers:** "How healthy is...", "Assess...", "Should we be concerned?"
1. **Genie** → Current readings and recent trend for the entity
2. **UC Function** → Call health assessment function with readings
3. **Return** → Health status, risk factors, and next actions

### Pattern D: Predictive / Forecasting
**Triggers:** "When will... fail?", "How long until...", "Forecast..."
1. **Genie** → Trend data (time-ordered, sufficient history)
2. **UC Function** → Call forecasting function with trend + threshold
3. **Return** → Predicted timeline, confidence, intervention window

### Pattern E: Benchmarking / Comparison
**Triggers:** "Compare...", "Which is better?", "Why is X worse than Y?"
1. **Genie** → KPIs for both entities (same time window)
2. **UC Function** → Call comparison function with both sets
3. **Synthesize** → Key differentiators, gaps, recommendations

### Pattern F: Direct Function Call
**Triggers:** User provides raw values directly in the question
1. **UC Function** → Call directly with provided values (no Genie needed)
2. **Return** → Parse result and present clearly

### Pattern G: Document-Only
**Triggers:** "What does the standard say?", "Best practice for...", "Procedure for..."
1. **AI Search** → Query the index
2. **Return** → Grounded answer citing the returned passages

### Pattern H: Discovery / Metadata
**Triggers:** "What tables exist?", "Describe...", "What functions are available?"
1. **SQL** → SHOW/DESCRIBE commands
2. **Return** → Formatted result

## Response guidelines
- Cite which tools you used and what data was retrieved.
- Use consistent units; round appropriately for the domain.
- Flag CRITICAL and HIGH risk conditions prominently.
- Never fabricate data values — always retrieve them.
- Include recommended actions (with urgency: immediate/scheduled/monitor) in every assessment.
- When presenting multiple entities, rank by severity/risk.
- State confidence level when making predictions.

## Error handling
- First call may cold-start (>30s) — retry once with 90s timeout.
- Genie error/no results → fall back to `sql-execution` with explicit SQL.
- One SQL statement per call — split multi-statement scripts.
- Function returns raw string → try JSON.parse; present raw if it fails.
- Entity not found → clarify the name; suggest similar entities from Genie.
- Permission denied → inform user; check USE CATALOG/SCHEMA/EXECUTE/SELECT grants.
"""

# --- help.md ---
help_md = f"""I'm a {DOMAIN_NAME.lower()} assistant. I can answer operational data
questions (Genie), run domain diagnostics and assessments (UC functions),
search reference documents and standards (AI Search), and explore the catalog (SQL).

### Ask data questions (Genie)
- "What events occurred on [entity] last week?"
- "Show me the trend for [metric] on [entity] over the past 30 days."
- "Which [entities] have the highest [metric]?"

### Compute & diagnose (UC functions)
- "Why is [entity] showing [symptom]? Diagnose the root cause."
- "Assess the health of [entity] given these readings."
- "When will [entity] reach the alarm threshold?"

### Reference documents (AI Search)
- "What does the standard say about [topic]?"
- "What's the recommended procedure for [situation]?"

### Discover the catalog (SQL)
- "What tables are available?" / "Describe [table_name]"

Type "help" or "what can you do?" any time to see this again.
"""

# --- skills/genie-data-questions/SKILL.md ---
# BEST PRACTICE: Include table schemas (column names + types) in the skill body.
# This helps the agent formulate better Genie questions and avoids hallucinated columns.
skill_genie = f"""---
name: genie-data-questions
description: >-
  Retrieve {DOMAIN_NAME.lower()} operational data from the scoped Databricks
  Genie space using natural language. PRIMARY path for any question needing
  rows from the database — events, measurements, time series, comparisons,
  history, trends, "which entities...", "show me...", "what happened...",
  "how many...". Genie writes and runs SQL for you.
allowed-tools: query_space_{space_id_placeholder} poll_response_{space_id_placeholder}
---

# Genie — {DOMAIN_NAME} Data (scoped space)

A scoped Databricks Genie space over your operational tables.
The primary way to retrieve data for this agent.

## Tables (include column names and types for better query formulation)
{chr(10).join(f'- `{t.split(".")[-1]}` — [KEY_COLUMNS: col1 (type), col2 (type), ...]' for t in DOMAIN_TABLES)}

> Tool names are space-scoped: `query_space_{space_id_placeholder}` and
> `poll_response_{space_id_placeholder}`, where the ID comes from
> `DATABRICKS_GENIE_MCP_SERVER_URL`.

## When to use
- Any question needing rows from the database.
- Also use Genie to fetch inputs for a UC function when the user gives none.

## Workflow
1. Ask with `query_space_...`; resolve vague time windows into explicit dates.
   Omit `conversation_id` on a new question.
2. If the response is in-progress, `poll_response_...` until complete.
3. Relay the result faithfully.

## Fallback
On error / no results / complex SQL, fall back to `sql-execution`.
For "what tables exist / describe X", use `sql-execution` directly.

## Output
Present the result table (all rows — no ellipsis) and the SQL when provided.
Never fabricate data values.
"""

# --- skills/uc-functions/SKILL.md ---
# BEST PRACTICE: The description MUST be exhaustive — include every function name,
# every trigger verb, every domain noun. This is the routing surface that determines
# whether the agent will even consider using your functions. Vague = skipped.
#
# If discovered_functions is populated (from auto-discovery above), we use the
# real names. Otherwise, we use placeholders that the next-steps tell you to replace.
if discovered_functions:
    fn_names_for_description = ', '.join(discovered_functions)
    fn_tools_line = ' '.join(f'{catalog_schema_tool}__{fn}' for fn in discovered_functions)
else:
    fn_names_for_description = '[REPLACE: list each function name, e.g. diagnose_issue, assess_health, ...]'
    fn_tools_line = f'{catalog_schema_tool}__<function_1> {catalog_schema_tool}__<function_2> {catalog_schema_tool}__<function_3>'

skill_functions = f"""---
name: uc-functions
description: >-
  Call Unity Catalog functions for {DOMAIN_NAME.lower()} computations. Functions
  include: {fn_names_for_description}. Use when
  the user asks "why", "what caused", "how healthy", "when will it fail",
  "compare", "should we quarantine/ship/hold", "diagnose", "assess", "forecast",
  "recommend", "classify", or needs expert interpretation of retrieved data —
  any computation beyond what SQL can answer.
allowed-tools: {fn_tools_line}
---

# Unity Catalog Functions

The Functions MCP server exposes each UC function under `{CATALOG}.{SCHEMA}` as a
tool named `{catalog_schema_tool}__<function>` (dots → double underscore).

## Available functions
[List your actual functions here with one-line descriptions]

## When to use
- User needs interpretation, diagnosis, risk assessment, predictions, or expert reasoning.
- User asks "why", "what caused", "how healthy", "when will", "compare".

## Retrieve-then-compute
When the user names an entity/window instead of giving raw numbers:
1. Retrieve the values with Genie (`genie-data-questions`).
2. Call the function per entity (not one aggregate).
3. Rank/synthesize and show the evidence.
Never feed a function guessed values.

## Output
State which function you called and the arguments. Relay the result verbatim;
add a one-line interpretation only if it helps the user act.
"""

# --- skills/vector-search-retrieval/SKILL.md ---
index_short_name = INDEX_NAME.split('.')[-1] if INDEX_NAME else "<index_name>"
skill_search = f"""---
name: vector-search-retrieval
description: AI Search (Databricks Vector Search) over {DOMAIN_NAME.lower()} reference documents and standards. Use for standards, best-practice, procedure, and policy questions — grounded in indexed documents rather than the operational database.
allowed-tools: {catalog_schema_tool}__{index_short_name}
---

# AI Search — {DOMAIN_NAME} Reference Documents (RAG)

The Vector Search server exposes the configured index as a retrieval tool.
Use it for document/knowledge questions, not for operational data (use Genie for data).

## Available index
- `{catalog_schema_tool}__{index_short_name}` — {DOMAIN_NAME.lower()} reference documents
  (diagnostics, standards, operations, safety).

## When to use
- "What does the standard say about [topic]?"
- "What's the recommended procedure for [situation]?"
- Any question about best practices, standards, or domain theory.

## Workflow
1. Query the index with the user's question (or a focused rephrasing).
2. Ground your answer in the returned chunks only; cite them.
3. If nothing relevant is returned, say so — don't use unsupported knowledge.

## Output
A grounded answer citing the passages; note which index you queried.
"""

# --- skills/sql-execution/SKILL.md ---
skill_sql = f"""---
name: sql-execution
description: Databricks SQL for (1) catalog/schema/table/function DISCOVERY via SHOW/DESCRIBE, and (2) a FALLBACK for complex queries or when Genie fails. For ordinary data retrieval prefer genie-data-questions; use this for metadata, user-provided SQL, or when Genie can't answer.
allowed-tools: execute_sql execute_sql_read_only poll_sql_result
---

# SQL Execution & Discovery (DBSQL)

The DBSQL MCP server executes SQL against Unity Catalog. Two roles:
metadata/discovery and a fallback for data retrieval (Genie is primary).

## Tools
- `execute_sql_read_only` — read-only query. **Default choice.**
- `execute_sql` — may modify data/state. Use only on explicit request; confirm first.
- `poll_sql_result` — poll a long-running statement by id.

## When to use
- Discovery: `SHOW CATALOGS` / `SHOW SCHEMAS IN <catalog>` /
  `SHOW TABLES IN <catalog>.<schema>` / `DESCRIBE TABLE <fqn>`.
- User-provided SQL, or a fallback when Genie errors / needs complex SQL.

## Workflow
1. Default to `execute_sql_read_only`.
2. Guard mutations — restate the change and confirm before any non-read-only SQL.
3. Fully qualify names as `catalog.schema.table`; backtick-quote identifiers with
   spaces. One statement per call.

## Output
Show the SQL, then the result as a complete Markdown table (all rows — no
ellipsis; truncation notes go after the table).
"""

# --- Save pack to volume ---
# Pack goes in a dedicated volume (not mixed with reference_docs)
PACK_DIR = f"{PACK_OUTPUT_PATH}/{domain_under}_pack"

print(f"{'='*70}")
print(f"GENERATING BEHAVIOR PACK: {DOMAIN_NAME}")
print(f"Location: {PACK_DIR}")
print(f"{'='*70}\n")

files = {
    "pack.yaml": pack_yaml,
    "system_prompt.md": system_prompt_md,
    "AGENTS.md": agents_md,
    "help.md": help_md,
    "skills/genie-data-questions/SKILL.md": skill_genie,
    "skills/uc-functions/SKILL.md": skill_functions,
    "skills/vector-search-retrieval/SKILL.md": skill_search,
    "skills/sql-execution/SKILL.md": skill_sql,
}

for filepath, content in files.items():
    full_path = f"{PACK_DIR}/{filepath}"
    dbutils.fs.put(full_path, content.strip(), overwrite=True)
    print(f"  ✅ {filepath} ({len(content.strip()):,} chars)")

print(f"\n{'='*70}")
print(f"PACK COMPLETE — {len(files)} files generated")
print(f"{'='*70}")
print(f"\nPack location: {PACK_DIR}")
print(f"\n{'='*70}")
print(f"NEXT STEPS (from best practices):")
print(f"{'='*70}")
print(f"  1. AGENTS.md: Replace placeholder thresholds with your real domain values")
print(f"  2. uc-functions/SKILL.md: Update the description with your real function names")
print(f"     AND update the allowed-tools line with real tool names.")
print(f"     Pattern: {CATALOG}__{SCHEMA}__<function_name>")
print(f"     ⚠️  allowed-tools is SPACE-separated (NOT commas!):")
print(f"        allowed-tools: {CATALOG}__{SCHEMA}__func_a {CATALOG}__{SCHEMA}__func_b")
print(f"  3. genie-data-questions/SKILL.md: Replace [KEY_COLUMNS: ...] with actual")
print(f"     column names and types from your tables")
print(f"  4. Replace <SPACE_ID> in genie skill if GENIE_SPACE_ID wasn't set")
print(f"  5. (Optional) Add skills/multi-tool-workflows/SKILL.md for domains with")
print(f"     many functions and complex multi-step patterns")
print(f"  6. Hand the pack folder + connection details to the deployment team")
print(f"\nDeployment env vars to provide:")
print(f"  DATABRICKS_AGENT_PACK_DIR=<mount_path>")
print(f"  DATABRICKS_GENIE_MCP_SERVER_URL=https://<host>/api/2.0/mcp/genie/{space_id_placeholder}")
print(f"  DATABRICKS_FUNCTIONS_MCP_SERVER_URL=https://<host>{FUNCTIONS_MCP_URL}")
print(f"  DATABRICKS_VECTORSEARCH_MCP_SERVER_URL=https://<host>{AI_SEARCH_MCP_URL}")
print(f"  DATABRICKS_DBSQL_MCP_SERVER_URL=https://<host>{SQL_MCP_URL}")
print(f"  DATABRICKS_OAUTH_CLIENT_ID={SERVICE_PRINCIPAL_ID}")
print(f"  DATABRICKS_OAUTH_CLIENT_SECRET=<secret>")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7812642126617340>, line 12
      9 import os
     11 # Derive names for the pack
---> 12 domain_slug = DOMAIN_NAME.lower().replace(' ', '-')
     13 domain_under = DOMAIN_NAME.lower().replace(' ', '_')
     14 space_id_placeholder = GENIE_SPACE_ID if GENIE_SPACE_ID else "<SPACE_ID>"

NameError: name 'DOMAIN_NAME' is not defined

# Step 10: Deploy the Behavior Pack

## Deployment Overview

Once you've generated and customized your behavior pack (Step 9), hand it to the deployment team. They mount the folder into the agent container and set environment variables pointing at your Databricks resources.

## Pack Folder Structure

```
my_domain_pack/                       ← your pack folder (any name)
├── pack.yaml                        ← manifest (agent card)
├── system_prompt.md                 ← persona + routing rules
├── AGENTS.md                        ← domain knowledge (memory)
├── help.md                          ← "what can you do?" text
└── skills/                          ← one sub-folder per capability
    ├── genie-data-questions/
    │   └── SKILL.md
    ├── uc-functions/
    │   └── SKILL.md
    ├── vector-search-retrieval/
    │   └── SKILL.md
    └── sql-execution/
        └── SKILL.md
```

## The 5 Required Files

| File | Purpose | Key Content |
|------|---------|-------------|
| `pack.yaml` | Agent card / manifest | Name, description, version, skill list |
| `system_prompt.md` | Persona + routing logic | Identity, 4 capabilities, decision order, guardrails, table-output policy |
| `AGENTS.md` | Durable domain knowledge | Terms, thresholds, formulas, composition patterns, response guidelines, error handling |
| `help.md` | Verbatim "help" text | Capability summary + starter prompts (shown as-is, model not invoked) |
| `skills/*.md` | Per-capability instructions | When/how to use each MCP server, tool names, workflows, fallbacks |

## Critical Rules for Skills

- The **sub-folder name MUST equal** the `name:` field in its `SKILL.md` frontmatter
- Folder/skill names: **lowercase letters, digits, and hyphens only** (1–64 chars)
- No leading/trailing `-`, no `--`
- `allowed-tools` is **space-separated** (NOT comma-separated!)
- Only `name` + `description` are loaded initially; the body is loaded **on demand** when the agent routes to that skill

## Tool Name Patterns

| Capability | Tool Name Pattern | Example |
|-----------|-------------------|--------|
| Genie | `query_space_<SPACE_ID>` `poll_response_<SPACE_ID>` | `query_space_01f19cc5...` |
| UC Functions | `<catalog>__<schema>__<function>` | `sf_demos__big_compute_demo__diagnose_equipment_event` |
| AI Search | `<catalog>__<schema>__<index>` | `sf_demos__big_compute_demo__energy_operations_reference_index` |
| SQL | `execute_sql` `execute_sql_read_only` `poll_sql_result` | (fixed names, always the same) |

> Note: dots become **double underscores** in tool names.

## How the Agent Uses Your Pack

```
Startup:
  1. Load pack.yaml → Set agent card (name/description)
  2. Load system_prompt.md → Set system prompt (routing + persona)
  3. Load AGENTS.md → Set domain knowledge reference
  4. Load help.md → Register help middleware
  5. Load skills/ → Index skill names + descriptions for routing
  6. Connect to 4 MCP servers via environment URLs
  7. Ready to serve

At query time:
  1. Match user intent against skill descriptions
  2. Load relevant skill body (on demand)
  3. Follow routing rules from system_prompt.md
  4. Execute tool calls via MCP
  5. Apply response guidelines from AGENTS.md
  6. Synthesize and return answer
```

## Deployment (Environment Variables)

Give the operator **two things**: your **pack folder** and the **connection details**. They do the rest.

### 1. The pack folder

Mounted into the server and selected via one env var:

```bash
# Pack location (or just mount the pack at /config/databricks_agent and omit this var)
DATABRICKS_AGENT_PACK_DIR=/config/databricks_agent

# MCP Server URLs
DATABRICKS_GENIE_MCP_SERVER_URL=https://<host>/api/2.0/mcp/genie/<SPACE_ID>
DATABRICKS_FUNCTIONS_MCP_SERVER_URL=https://<host>/api/2.0/mcp/functions/<catalog>/<schema>
DATABRICKS_VECTORSEARCH_MCP_SERVER_URL=https://<host>/api/2.0/mcp/ai-search/<catalog>/<schema>
# ^ Schema-scoped (not index-scoped). Auto-discovers all indexes in the schema.
#   The tool name still includes the index: {catalog}__{schema}__{index_name}
DATABRICKS_DBSQL_MCP_SERVER_URL=https://<host>/api/2.0/mcp/sql

# Authentication
DATABRICKS_OAUTH_CLIENT_ID=<service-principal-app-id>
DATABRICKS_OAUTH_CLIENT_SECRET=<secret>  # kept as a secret, never in the pack
```

> **Mounting note:** A pack with a nested `skills/<id>/SKILL.md` subtree must be mounted from a **volume/PVC** (or baked into an image). A Kubernetes ConfigMap is flat and **cannot** hold the `skills/` subtree, so ConfigMap-only works just for a single-level pack. Use a volume for a full pack.

## Pre-Deploy Validation Checklist

Before handing off, verify:

- [ ] Pack folder contains **`pack.yaml`, `system_prompt.md`, `AGENTS.md`, `help.md`, and `skills/`**
- [ ] Every `skills/<id>/` has a **`SKILL.md`**, and each `SKILL.md`'s `name:` **exactly equals** its folder name
- [ ] Skill/folder names are **lowercase-hyphen** only (no leading/trailing `-`, no `--`)
- [ ] Each `SKILL.md` has a **`description`** rich with domain keywords (1–1024 chars)
- [ ] `allowed-tools` values are **space-separated** (no commas)
- [ ] `pack.yaml` is **valid YAML** and lists the skills you shipped
- [ ] `system_prompt.md` includes a **no-ellipsis table policy** and the read-only-by-default rule
- [ ] No made-up tool outputs or numbers anywhere in the text
- [ ] You've written down the **four resource URLs** + the **service principal** to hand to the operator

## Verification After Deploy

- Agent card at `…/a2a/databricks_agent/.well-known/agent-card.json` shows YOUR name/description (from `pack.yaml`)
- Typing "help" returns YOUR `help.md` text verbatim
- A data question triggers Genie; a computation triggers UC Functions

In [0]:
# =============================================================================
# TEST MCP SERVER CONNECTIVITY
# =============================================================================
# Verifies each MCP server responds correctly.
# Uses the notebook's own token (not the service principal).
# =============================================================================

import requests
import json

WORKSPACE_HOST = spark.conf.get("spark.databricks.workspaceUrl")
BASE_URL = f"https://{WORKSPACE_HOST}"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

def test_mcp_server(name, endpoint):
    """Initialize an MCP session and list tools."""
    url = f"{BASE_URL}{endpoint}"
    print(f"\n{'─'*50}")
    print(f"Testing: {name}")
    print(f"  URL: {url}")
    
    # Step 1: Initialize
    init_payload = {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "blueprint-test", "version": "1.0.0"}
    }}
    
    try:
        resp = requests.post(url, headers=headers, json=init_payload, timeout=30)
        if resp.status_code != 200:
            print(f"  ❌ Initialize failed: HTTP {resp.status_code}")
            print(f"     {resp.text[:150]}")
            return False
        
        session_id = resp.headers.get("Mcp-Session-Id", "")
        if not session_id:
            print(f"  ⚠️  No Mcp-Session-Id in response headers")
            return False
        
        print(f"  ✅ Initialize OK (session: {session_id[:20]}...)")
        
        # Step 2: List tools
        list_headers = {**headers, "Mcp-Session-Id": session_id}
        list_payload = {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
        
        resp2 = requests.post(url, headers=list_headers, json=list_payload, timeout=30)
        if resp2.status_code == 200:
            result = resp2.json()
            tools = result.get("result", {}).get("tools", [])
            print(f"  ✅ tools/list OK ({len(tools)} tool(s) available)")
            for tool in tools[:5]:  # Show first 5
                print(f"     • {tool.get('name', 'unnamed')}")
            if len(tools) > 5:
                print(f"     ... and {len(tools) - 5} more")
            return True
        else:
            print(f"  ❌ tools/list failed: HTTP {resp2.status_code}")
            return False
            
    except requests.exceptions.Timeout:
        print(f"  ❌ Timeout (server may need cold start — retry)")
        return False
    except Exception as e:
        print(f"  ❌ Error: {str(e)[:100]}")
        return False

# --- Run Tests ---
print("="*50)
print("MCP SERVER CONNECTIVITY TEST")
print("="*50)

results = {}

# Test UC Functions
results["UC Functions"] = test_mcp_server("UC Functions", FUNCTIONS_MCP_URL)

# Test Genie Space
if GENIE_SPACE_ID:
    results["Genie Space"] = test_mcp_server("Genie Space", f"/api/2.0/mcp/genie/{GENIE_SPACE_ID}")
else:
    print(f"\n{'─'*50}")
    print("Testing: Genie Space")
    print("  ⚠️  Skipped (GENIE_SPACE_ID not set)")
    results["Genie Space"] = None

# Test AI Search
results["AI Search"] = test_mcp_server("AI Search", AI_SEARCH_MCP_URL)

# Test SQL
results["Databricks SQL"] = test_mcp_server("Databricks SQL", SQL_MCP_URL)

# Summary
print(f"\n{'='*50}")
print("RESULTS SUMMARY")
print(f"{'='*50}")
for name, status in results.items():
    icon = "✅" if status else ("⚠️" if status is None else "❌")
    print(f"  {icon} {name}")

passed = sum(1 for v in results.values() if v is True)
total = sum(1 for v in results.values() if v is not None)
print(f"\n  {passed}/{total} servers responding")

In [0]:
# =============================================================================
# SUMMARY: YOUR MCP SERVER ENDPOINTS
# =============================================================================

WORKSPACE_HOST = spark.conf.get("spark.databricks.workspaceUrl")

print(f"""
{'='*70}
✅ MCP SERVER SETUP COMPLETE: {DOMAIN_NAME}
{'='*70}

Your 4 MCP servers are ready for the Spotfire Agent:

┌─────────────────┬───────────────────────────────────────────────────────────────────────┐
│ Server            │ Endpoint                                                             │
├─────────────────┼───────────────────────────────────────────────────────────────────────┤
│ UC Functions      │ {FUNCTIONS_MCP_URL:<69}│
│ Genie Space       │ {('/api/2.0/mcp/genie/' + GENIE_SPACE_ID if GENIE_SPACE_ID else '<set GENIE_SPACE_ID>'):<69}│
│ AI Search         │ {AI_SEARCH_MCP_URL:<69}│
│ Databricks SQL    │ {SQL_MCP_URL:<69}│
└─────────────────┴───────────────────────────────────────────────────────────────────────┘

Full base URL: https://{WORKSPACE_HOST}

{'='*70}
NEXT STEPS
{'='*70}

1. ✅ Run Cell 18 to generate your behavior pack (saved to volume)
2. ✅ Customize the pack files (especially AGENTS.md domain knowledge)
3. ✅ Export the pack from the UC Volume to your local filesystem:
     databricks fs cp -r dbfs:{PACK_OUTPUT_PATH}/{DOMAIN_NAME.lower().replace(' ', '_')}_pack ./my_pack/
     (Or download via Catalog → Volumes UI — see README for all options)
4. ✅ Hand the exported folder + these env vars to the deployment team:
     - DATABRICKS_AGENT_PACK_DIR=<mount_path>
     - DATABRICKS_GENIE_MCP_SERVER_URL=https://{WORKSPACE_HOST}/api/2.0/mcp/genie/{GENIE_SPACE_ID if GENIE_SPACE_ID else '<SPACE_ID>'}
     - DATABRICKS_FUNCTIONS_MCP_SERVER_URL=https://{WORKSPACE_HOST}{FUNCTIONS_MCP_URL}
     - DATABRICKS_VECTORSEARCH_MCP_SERVER_URL=https://{WORKSPACE_HOST}{AI_SEARCH_MCP_URL}
     - DATABRICKS_DBSQL_MCP_SERVER_URL=https://{WORKSPACE_HOST}{SQL_MCP_URL}
     - DATABRICKS_OAUTH_CLIENT_ID={SERVICE_PRINCIPAL_ID}
     - DATABRICKS_OAUTH_CLIENT_SECRET=<secret>
5. ✅ Verify: agent card shows your name, "help" returns your text
6. ✅ Test end-to-end with a sample question

{'='*70}
ARCHITECTURE RECAP
{'='*70}

  User Question
       │
       ▼
  Spotfire Agent (reads SKILL.md + Agents.md)
       │
       ├─── Genie MCP ─── (retrieves data)
       ├─── UC Functions MCP ─── (interprets data)
       ├─── AI Search MCP ─── (retrieves knowledge)
       └─── SQL MCP ─── (discovery & fallback)
       │
       ▼
  Expert Answer
""")

# Appendix A: OAuth M2M Authentication for External Clients

## How External Clients Authenticate

The Spotfire Agent (or any MCP client) authenticates to Databricks using **OAuth Machine-to-Machine (M2M)** with a service principal.

## OAuth Token Flow

```
1. Client sends client_id + client_secret to token endpoint
2. Token endpoint returns an access_token (expires in ~1 hour)
3. Client includes access_token in Authorization header on MCP requests
4. When token expires, client requests a new one (refresh flow)
```

## Token Endpoint

```
POST https://{{workspace}}/oidc/v1/token
Content-Type: application/x-www-form-urlencoded

grant_type=client_credentials
&client_id={{service_principal_uuid}}
&client_secret={{your_secret}}
&scope=all-apis
```

## Complete MCP Flow (curl)

```bash
# --- Variables ---
WORKSPACE="your-workspace.azuredatabricks.net"
CLIENT_ID="00000000-0000-0000-0000-000000000000"
CLIENT_SECRET="your-secret-value"
MCP_ENDPOINT="/api/2.0/mcp/functions/my_catalog/my_schema"

# --- Step 1: Get OAuth Token ---
TOKEN=$(curl -s -X POST "https://${WORKSPACE}/oidc/v1/token" \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "grant_type=client_credentials&client_id=${CLIENT_ID}&client_secret=${CLIENT_SECRET}&scope=all-apis" \
  | python3 -c "import sys,json; print(json.load(sys.stdin)['access_token'])")

echo "Token obtained: ${TOKEN:0:20}..."

# --- Step 2: Initialize MCP Session ---
RESPONSE=$(curl -s -D /tmp/mcp_headers -X POST "https://${WORKSPACE}${MCP_ENDPOINT}" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
      "protocolVersion": "2025-03-26",
      "capabilities": {},
      "clientInfo": {"name": "my-client", "version": "1.0.0"}
    }
  }')

# Extract session ID from response headers
SESSION_ID=$(grep -i "mcp-session-id" /tmp/mcp_headers | cut -d: -f2 | tr -d ' \r')
echo "Session ID: ${SESSION_ID}"

# --- Step 3: Send initialized notification ---
curl -s -X POST "https://${WORKSPACE}${MCP_ENDPOINT}" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Mcp-Session-Id: ${SESSION_ID}" \
  -d '{"jsonrpc": "2.0", "method": "notifications/initialized", "params": {}}'

# --- Step 4: List Available Tools ---
curl -s -X POST "https://${WORKSPACE}${MCP_ENDPOINT}" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Mcp-Session-Id: ${SESSION_ID}" \
  -d '{
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
  }' | python3 -m json.tool

# --- Step 5: Call a Tool ---
curl -s -X POST "https://${WORKSPACE}${MCP_ENDPOINT}" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Mcp-Session-Id: ${SESSION_ID}" \
  -d '{
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
      "name": "my_catalog__my_schema__diagnose_issue",
      "arguments": {
        "entity_id": "EQUIP-001",
        "entity_type": "compressor",
        "issue_description": "High vibration detected",
        "severity": "High",
        "duration": 2.5
      }
    }
  }' | python3 -m json.tool
```

## Key Points

| Aspect | Detail |
|--------|--------|
| Token lifetime | ~1 hour (refresh before expiry) |
| Session scope | One session per MCP endpoint |
| Header name | `Mcp-Session-Id` (case-sensitive) |
| Tool name format | `catalog__schema__function_name` (double underscores) |
| Protocol version | `2025-03-26` (or latest) |
| Transport | Streamable HTTP (single URL, POST only) |

# Appendix B: Troubleshooting

## Common Issues and Fixes

### Permission Errors

| Error | Cause | Fix |
|-------|-------|-----|
| `PERMISSION_DENIED: User does not have USE CATALOG` | SP missing catalog grant | `GRANT USE CATALOG ON CATALOG {{catalog}} TO \`{{sp_id}}\`` |
| `PERMISSION_DENIED: User does not have EXECUTE` | SP missing execute grant | `GRANT EXECUTE ON SCHEMA {{catalog}}.{{schema}} TO \`{{sp_id}}\`` |
| `PERMISSION_DENIED: User does not have SELECT` | SP missing select grant | `GRANT SELECT ON SCHEMA {{catalog}}.{{schema}} TO \`{{sp_id}}\`` |
| `Genie: permission denied` | SP not added to Genie space | Grant CAN_RUN via UI or API (Step 8) |
| `403 on MCP endpoint` | OAuth token expired or invalid scope | Refresh token; ensure `scope=all-apis` |

### Function Timeouts

| Symptom | Cause | Fix |
|---------|-------|-----|
| First call takes 30-60s | Cold start (model loading) | Normal — retry once; subsequent calls are fast |
| All calls timeout (>120s) | Model endpoint down | Check endpoint status in Model Serving UI |
| Intermittent timeouts | Token budget exceeded | Reduce `max_tokens` or simplify prompt |

### Genie Issues

| Symptom | Cause | Fix |
|---------|-------|-----|
| Genie returns empty results | Wrong table or misspelled entity | Verify entity names exist in the data; add sample questions |
| Genie misinterprets question | Ambiguous query | Add more sample questions in the Genie space UI |
| Genie SQL error | Complex query beyond Genie's capability | Fall back to Databricks SQL MCP |

### AI Search Issues

| Symptom | Cause | Fix |
|---------|-------|-----|
| Returns irrelevant chunks | Poor chunking | Reduce `MAX_CHUNK_CHARS`; improve section boundaries |
| Returns nothing | Index not synced | `w.vector_search_indexes.sync_index(index_name=INDEX_NAME)` |
| Index sync fails | Missing CDC on table | `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)` |
| Embedding errors | Model endpoint unavailable | Check `databricks-gte-large-en` endpoint status |

### MCP Session Errors

| Symptom | Cause | Fix |
|---------|-------|-----|
| `Session not found` | Missing `Mcp-Session-Id` header | Include the header from initialize response on ALL requests |
| `Invalid session` | Session expired | Re-initialize (sessions expire after inactivity) |
| `Method not found` | Wrong method name | Use `tools/list`, `tools/call` (not `tool/list`) |

### JSON Parse Errors

| Symptom | Cause | Fix |
|---------|-------|-----|
| Function returns markdown code fences | Model ignoring JSON instruction | Strengthen prompt: "Output raw JSON starting with { and ending with }. Do not include any explanation or markdown code fences." |
| Truncated JSON | max_tokens too low | Increase `max_tokens` parameter |
| Extra text after JSON | Model chatty | Post-process: extract first `{...}` block |

### UC Function Creation Errors

| Error | Cause | Fix |
|-------|-------|-----|
| `NOT_A_VALID_DEFAULT_PARAMETER_POSITION` | DEFAULT param before required param | Move all DEFAULT params to the end |
| `ai_query model not found` | Wrong model name | Check model endpoint name in Model Serving |
| `AMBIGUOUS_REFERENCE` | Unqualified column in JOIN | Always use `table_alias.column` syntax |

---

## Useful Diagnostic Commands

```sql
-- Check function exists and has metadata
DESCRIBE FUNCTION EXTENDED {{catalog}}.{{schema}}.{{function_name}};

-- List all functions in schema
SHOW FUNCTIONS IN {{catalog}}.{{schema}};

-- Test a function directly
SELECT {{catalog}}.{{schema}}.diagnose_issue(
    'TEST-001', 'compressor', 'High vibration', 'High', 2.5
) AS result;

-- Check Vector Search index status
SELECT * FROM system.information_schema.vector_indexes
WHERE index_name = '{{index_name}}';
```

```python
# Check index sync status
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
idx = w.vector_search_indexes.get_index(index_name="{{catalog}}.{{schema}}.{{index_name}}")
print(f"Status: {idx.status}")
print(f"Sync state: {idx.delta_sync_index_spec.pipeline_status if idx.delta_sync_index_spec else 'N/A'}")
```